# Урок 21. Метрики классификации: почему одной accuracy мало

**Дата:** 11.02.2027 · **60 минут** · 8–9 класс  
**Модуль 4.** Машинное обучение (уроки 17–25)  
**Опирается на:** урок 19 (train / test, accuracy), урок 20 (сравнение моделей, разбор ошибок).

Модель угадывает 91% ответов — это хорошо? Ответ: **смотря на чём ошибается**. Сегодня мы увидим модель, которая не делает ничего полезного, но набирает 85%, и научимся считать ошибки по видам: **матрица ошибок**, **precision** и **recall**. Это главный инструмент, чтобы понять, можно ли модели доверять.

## Цели урока

К концу занятия ты сможешь:

1. Сравнивать модель с **базовой моделью** (baseline).
2. Строить и читать **матрицу ошибок**: TP, FP, FN, TN.
3. Считать и объяснять **precision** и **recall**.
4. Выбирать главную метрику по **цене ошибки** в задаче.
5. Менять **порог** модели и видеть, как меняются метрики.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–10 | Задача и ловушка «ленивой модели» |
| 10–25 | Матрица ошибок |
| 25–35 | Precision, recall, цена ошибки |
| 35–42 | Порог модели |
| 42–55 | Практика |
| 55–60 | Итог и домашнее задание |

## 1. Задача: кто может бросить кружок

Куратор хочет **заранее** узнать, кто из учеников может бросить кружок, чтобы поговорить с ними и помочь. Признаки: посещаемость (%), сколько занятий подряд пропущено сейчас, сколько из 10 домашних заданий сдано. Цель: `бросил` (да / нет).

Класс, который мы **ищем** — `бросил = True`, — называют **положительным** (*positive*). Это не значит «хороший»: в медицине положительный класс — «болен», в почте — «спам».

In [ ]:
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def make_club(n, seed):
    """Учебные данные: кто бросит кружок."""
    random.seed(seed)
    rows = []
    for i in range(n):
        attendance = random.randint(40, 100)
        missed = random.randint(0, 6)
        homework = random.randint(0, 10)
        risk = -0.75 + 0.015 * (100 - attendance) + 0.07 * missed + 0.02 * (10 - homework)
        rows.append({
            "посещаемость": attendance,
            "пропусков_подряд": missed,
            "сдано_дз": homework,
            "бросил": random.random() < risk,
        })
    return pd.DataFrame(rows)

df = make_club(400, seed=22)
print(df["бросил"].value_counts())
print("Доля бросивших:", round(df["бросил"].mean(), 3))

Бросают кружок около 15% учеников — это **редкий** класс. Такие задачи называют задачами с **несбалансированными классами**. Они встречаются постоянно: болезни, мошенничество, поломки, спам — всё «плохое» обычно редкое.

In [ ]:
X = df[["посещаемость", "пропусков_подряд", "сдано_дз"]]
y = df["бросил"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_train, y_train)
pred = tree.predict(X_test)
print("Accuracy дерева на test:", accuracy_score(y_test, pred))

## 2. Ловушка: «ленивая» модель

А теперь модель, которая **ничего не изучает** и всем отвечает одно и то же — самый частый класс («не бросит»). В sklearn такая есть: `DummyClassifier` («модель-пустышка»).

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
print("Accuracy «ленивой» модели:", accuracy_score(y_test, dummy_pred))
print("Сколько учеников она предупредила о риске:", dummy_pred.sum())

85% правильных ответов — и **ни одного** найденного ученика, который бросит! Для куратора такая модель бесполезна: ровно тех, кому нужна помощь, она не замечает.

Вывод 1: **accuracy обманывает, когда классы несбалансированы**.

Вывод 2: любую модель сравнивают с **базовой** (*baseline*) — самым простым решением. Наше дерево (0.91) лучше пустышки (0.85) всего на 6 процентных пунктов (91% − 85%). Насколько оно полезно? Нужно смотреть глубже.

## 3. Матрица ошибок

Разложим ответы модели на четыре вида:

| | Модель: **бросит** | Модель: **не бросит** |
|--|:--:|:--:|
| **Правда: бросил** | **TP** — верная тревога | **FN** — пропуск |
| **Правда: не бросил** | **FP** — ложная тревога | **TN** — верное спокойствие |

- **TP** (*true positive*) — нашли того, кого искали.
- **FN** (*false negative*) — **пропустили**: человек бросил, а модель промолчала.
- **FP** (*false positive*) — **ложная тревога**: модель забила тревогу зря.
- **TN** (*true negative*) — верно сказали «всё в порядке».

`confusion_matrix` в sklearn выдаёт эти числа таблицей. Для ответов `False / True` порядок такой: `[[TN, FP], [FN, TP]]`.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
print(cm)

TN = cm[0][0]
FP = cm[0][1]
FN = cm[1][0]
TP = cm[1][1]
print(f"TP = {TP}, FN = {FN}, FP = {FP}, TN = {TN}")

`cm[0][0]` — как `rows[0]["score"]` на уроке 8: первое число — номер строки таблицы, второе — номер столбца.

Читаем: в test 15 учеников, которые бросили. Модель нашла только **7** (TP), а **8** пропустила (FN). Ложная тревога всего одна (FP). Высокая accuracy получилась в основном за счёт TN — «спокойных» учеников, которых большинство.

### Посчитай вручную

Модель проверки спама (положительный класс — «спам») ответила на 10 писем:

| Письмо | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|--------|---|---|---|---|---|---|---|---|---|----|
| Правда | спам | спам | спам | норм | норм | норм | норм | норм | спам | норм |
| Модель | спам | норм | спам | норм | норм | спам | норм | норм | спам | норм |

Сколько TP, FN, FP, TN?

<details>
<summary>Ответ</summary>

TP = 3 (письма 1, 3, 9), FN = 1 (письмо 2 — спам пропущен), FP = 1 (письмо 6 — нормальное письмо попало в спам), TN = 5.
</details>

## 4. Precision и recall

Из матрицы ошибок получают две главные метрики.

**Recall** (полнота) — какую долю **искомых** мы нашли:

$$\text{recall} = \frac{TP}{TP + FN}$$

«Из всех, кто бросил, скольких модель заметила?»

**Precision** (точность) — какая доля **тревог** была по делу:

$$\text{precision} = \frac{TP}{TP + FP}$$

«Из всех, о ком модель предупредила, сколько действительно бросили?»

In [ ]:
from sklearn.metrics import precision_score, recall_score

print("Recall:", round(recall_score(y_test, pred), 2), "| вручную:", round(TP / (TP + FN), 2))
print("Precision:", round(precision_score(y_test, pred), 2), "| вручную:", round(TP / (TP + FP), 2))

Recall 0.47 — модель находит меньше половины тех, кто бросит. Precision 0.88 — зато если она подняла тревогу, то почти всегда по делу.

> **Аналогия с рыбалкой.** Recall — какую долю рыбы в озере ты поймал. Precision — какая доля улова — рыба, а не старые ботинки. Сеть с крупными ячейками ловит мало мусора (precision высокая), но много рыбы уходит (recall низкий).

Иногда precision и recall объединяют в одно число **F1** — среднее, которое мало, если мала хотя бы одна из метрик. Но смысл лучше видно, когда смотришь на обе метрики отдельно.

## 5. Цена ошибки: какая метрика важнее

Какая ошибка хуже — FN или FP? Это зависит от **задачи**, а не от математики.

| Задача | FN (пропуск) | FP (ложная тревога) | Что важнее |
|--------|--------------|---------------------|------------|
| Кто бросит кружок | не помогли ученику, он ушёл | лишний разговор с учеником | **recall** |
| Спам-фильтр | спам в почте — неприятно | важное письмо в спаме — плохо | **precision** |
| Ранняя диагностика болезни | болезнь не заметили — опасно | лишнее обследование | **recall** |
| Блокировка аккаунта за мошенничество | мошенник прошёл | честного человека заблокировали | баланс, решают люди |

В нашей задаче ложная тревога стоит недорого — дружеский разговор. А пропуск — ученик уходит из кружка. Значит, нам важнее **recall**: лучше предупредить лишний раз, чем пропустить.

## 6. Порог: как сдвинуть баланс

Модель на самом деле может выдать не просто «да / нет», а **вероятность** класса. Откуда она у дерева? Каждый новый ученик, ответив на вопросы дерева, попадает в одну из «конечных» групп — **лист** дерева. Дерево смотрит, какая доля обучающих учеников в этом листе бросила кружок: если 3 из 4 — вероятность 0.75.

Метод `predict_proba` показывает эти вероятности. `predict` отвечает «да», если вероятность ≥ 0.5 — это **порог**.

In [ ]:
proba = tree.predict_proba(X_test)
print(proba[:5])

`proba[:5]` — первые 5 строк (срез из урока 6). В каждой строке два числа: вероятность «не бросит» и вероятность «бросит», в сумме 1. Нам нужен второй столбец — `proba[:, 1]` (все строки, столбец с индексом 1).

Порог можно **сдвинуть**. Ниже порог — модель чаще поднимает тревогу: recall растёт, precision падает.

In [ ]:
risk = proba[:, 1]
for threshold in [0.5, 0.3, 0.2]:
    alarm = risk >= threshold
    p = precision_score(y_test, alarm)
    r = recall_score(y_test, alarm)
    print(f"порог {threshold}: тревог {alarm.sum():2}, precision {p:.2f}, recall {r:.2f}")

С порогом 0.2 модель находит **80%** будущих «бросивших» — ценой того, что больше половины тревог ложные. Для куратора это хороший обмен: разговоров больше, зато почти никого не пропустим.

**Порог — это решение человека**, а не модели. Инженер показывает, как меняются метрики, а люди решают, какие ошибки допустимы. Честно: порог тоже нельзя долго подбирать, глядя на test, — это то же подглядывание. Как делать правильно — на уроке 23.

## 7. Карточка метрик

После любой классификации заполняй карточку — она понадобится в итоговом проекте.

| Пункт | Пример |
|-------|--------|
| Задача | заранее найти, кто может бросить кружок |
| Положительный класс | бросил |
| Baseline | «всем — не бросит»: accuracy 0.85, recall 0 |
| Модель | дерево глубины 3, порог 0.2 |
| Главная метрика и почему | recall: пропуск дороже ложной тревоги |
| Результат | recall 0.80, precision 0.39 |
| Что это значит словами | находим 4 из 5 будущих ушедших; из 5 тревог примерно 2 по делу |

## 8. Практика

### Практика 1. Спам-фильтр в коде (6 минут)

Проверь свой ручной подсчёт из раздела 3 кодом: построй матрицу ошибок, precision и recall. Положительный класс — спам (`True`).

In [ ]:
truth = [True, True, True, False, False, False, False, False, True, False]
model = [True, False, True, False, False, True, False, False, True, False]
# Практика 1

<details>
<summary>Решение</summary>

```python
print(confusion_matrix(truth, model))
print("precision:", precision_score(truth, model))
print("recall:", recall_score(truth, model))
```
Precision и recall равны 0.75: из 4 тревог 3 по делу, из 4 спам-писем найдено 3.
</details>

### Практика 2. Сравни с kNN (8 минут)

Обучи kNN (k = 5) на этих данных — не забудь масштабирование по train из урока 20. Посчитай accuracy, precision и recall. Какую модель ты бы выбрал для куратора и почему?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# Практика 2

<details>
<summary>Решение</summary>

```python
max_train = X_train.max()
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train / max_train, y_train)
knn_pred = knn.predict(X_test / max_train)
print("accuracy:", round(accuracy_score(y_test, knn_pred), 2))
print("precision:", round(precision_score(y_test, knn_pred, zero_division=0), 2))
print("recall:", round(recall_score(y_test, knn_pred), 2))
```
Сравнивай модели по **главной метрике задачи** (recall), а не только по accuracy. `zero_division=0` нужен на случай, если модель не подняла ни одной тревоги — тогда precision не из чего считать.
</details>

### Практика 3. Какая метрика главная? (6 минут)

Для каждой задачи назови положительный класс, что такое FN и FP, и какая метрика важнее.

1. Система, которая предупреждает о пожаре в здании.
2. Рекомендация «это видео тебе понравится».
3. Автоматическая проверка: списал ли ученик работу.

1. …
2. …
3. …

<details>
<summary>Ответы</summary>

1. Положительный класс — пожар. FN — пожар не заметили (катастрофа), FP — ложная тревога (неприятно). Главная — **recall**.
2. Положительный — «понравится». FN — не показали хорошее видео (небольшая потеря), FP — показали неинтересное (тоже небольшая). Обычно важнее **precision**: зритель видит только рекомендованное.
3. Положительный — «списал». FP — обвинили честного ученика (очень плохо), FN — списавший не пойман. Главная — **precision**, а окончательное решение всегда за человеком.
</details>

## Работа с ИИ-помощником

- «Объясни разницу между precision и recall на примере пожарной сигнализации.»
- «Проверь мою карточку метрик: правильно ли я выбрал главную метрику?»
- Помни: если помощник выдаёт таблицу метрик, сверь её с выводом своего кода.

## Мини-глоссарий урока 21

| Термин | Смысл |
|--------|-------|
| **Baseline** | простейшее решение для сравнения |
| **Несбалансированные классы** | один класс намного реже другого |
| **Положительный класс** | класс, который мы ищем |
| **TP / FP / FN / TN** | верная тревога / ложная тревога / пропуск / верное спокойствие |
| **Матрица ошибок** | таблица этих четырёх чисел |
| **Recall (полнота)** | доля найденных среди всех искомых |
| **Precision (точность)** | доля верных среди всех тревог |
| **Порог** | вероятность, начиная с которой модель говорит «да» |

## Итог урока

1. Сначала сравни модель с baseline: высокая accuracy может быть у бесполезной модели.
2. Матрица ошибок показывает, **какие** ошибки делает модель; precision и recall — их доли.
3. Главную метрику выбирают по **цене ошибки** в реальной задаче; порог — решение людей.

## Домашнее задание (20 минут)

Придумай свою задачу классификации с редким классом (не из урока). Заполни для неё карточку метрик: положительный класс, что означают FN и FP **в жизни**, какая ошибка хуже, какая метрика главная, как бы ты выбрал порог.

**Моя карточка метрик:** …

## Самопроверка

1. Как «ленивая» модель может иметь accuracy 85%?
2. Что такое FN в задаче о бросивших кружок?
3. Чем precision отличается от recall?
4. Что произойдёт с recall, если снизить порог?
5. Почему выбор порога — решение людей, а не модели?

**Следующий урок (18.02.2027):** регрессия — когда нужно предсказать число.